# Tox21 Molecular Toxicity Prediction: Reproducible Feature Engineering Pipeline
**Phase**: Member 2 Final Deliverable (Handoff to Member 3)  
**Role**: Senior ML Engineer & Cheminformatics Specialist  
**Upstream**: Member 1 (Dataset Curation, Invalid SMILES Removal, Corrected Scaffold Split)  
**Downstream**: Member 3 (Baseline Model Training, Cross-Validation, Benchmark Evaluation)  

---

## 1. Project Scope, Assumptions, and Reproducibility

### 1.1 Objective & Context
The Tox21 Data Challenge established a benchmark of ~10,000 environmental chemicals evaluated across 12 nuclear receptor (NR) and stress response (SR) toxicity assays. In this multi-member team project:
- **Member 1** audited the raw dataset, filtered 8 unparseable/invalid molecular structures, canonicalized chemical SMILES, generated Bemis–Murcko scaffolds, and corrected the scaffold split to produce an official **80.01% train, 10.00% validation, and 10.00% test** partition.
- **Member 2 (This Pipeline)** extracts molecular representations (Morgan circular fingerprints and physicochemical descriptors), performs leakage-safe transformations and dimensionality reduction (PCA, variance filtering, supervised univariate associations), and exports clean, certified artifacts.
- **Member 3** ingests these exported matrices and observed-label masks to train and evaluate baseline classification models (e.g., Logistic Regression, Random Forest, Gradient Boosting, GNNs) without manual upstream preprocessing.

### 1.2 Core Reproducibility Rules & Leakage Boundaries
1. **Frozen Source of Truth**: Reads exclusively from the audited and validated dataset (`phase1_audit/split_correction/corrected_scaffold_split.csv`).
2. **Stable Molecule ID Alignment**: Molecule IDs (`mol_id`, `TOX...`) and row positions are strictly aligned across all feature tables, label arrays, and split masks. Features and labels are never independently sorted.
3. **Strict Leakage Protection**: All statistical preprocessing parameters (descriptor scaling mean/std, PCA eigenvectors, unsupervised variance thresholds, supervised univariate rankings) are fitted **STRICTLY on the training split** ($N = 6,259$). Validation ($N = 782$) and test ($N = 782$) splits are transformed using frozen training objects. Target labels are never consulted during feature generation or unsupervised reductions.
4. **Missing Label Integrity**: Missing toxicity assay outcomes are preserved as IEEE 754 `NaN`. No missing label is imputed as negative (`0.0`), preventing experimental sparsity from biasing predictive signals.
5. **Deterministic Execution**: All stochastic operations and transformations use fixed random seeds (`RANDOM_SEED = 42`).

In [1]:
import os
import sys
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator

# Add handoff directory to python path for features module
HANDOFF_DIR = Path.cwd()
if str(HANDOFF_DIR) not in sys.path:
    sys.path.append(str(HANDOFF_DIR))

import features as ft

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Python Environment Initialized.")
print(f"Working Directory : {HANDOFF_DIR}")
print(f"RDKit Version     : {rdBase.rdkitVersion}")
print(f"Pandas Version    : {pd.__version__}")
print(f"NumPy Version     : {np.__version__}")

Python Environment Initialized.
Working Directory : /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff
RDKit Version     : 2026.09.1
Pandas Version    : 2.3.3
NumPy Version     : 2.2.6


## 2. Environment and Dependency Checks

Verifying local software versions, directory structure, and import integrity before executing downstream transformations.

In [2]:
def verify_environment():
    assert rdkit.__version__ is not None, "RDKit must be installed"
    assert int(pd.__version__.split('.')[0]) >= 1, "Pandas >= 1.0 required"
    assert int(np.__version__.split('.')[0]) >= 1, "NumPy >= 1.20 required"
    
    artifacts_dir = HANDOFF_DIR / "artifacts"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    print(f"Artifacts output directory ready: {artifacts_dir}")
    return True

verify_environment()

Artifacts output directory ready: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/artifacts


## 3. Loading the Dataset and Explaining its Provenance

### Provenance Chain
1. **Raw Tox21 File** (`tox21.csv`): 7,831 raw records with compound IDs, raw SMILES, and 12 toxicity endpoints.
2. **Member 1 Audit & Cleaning**:
   - Evaluated chemical validity with RDKit sanitization.
   - Identified 8 invalid/unparseable SMILES (organometallics, invalid valences).
   - Removed the 8 invalid rows, yielding 7,823 valid compounds.
   - Canonicalized SMILES using `Chem.MolToSmiles(mol, canonical=True)`.
   - Extracted Bemis–Murcko scaffolds (`Chem.Scaffolds.MurckoScaffold.MurckoScaffoldSmiles`).
3. **Scaffold Split Resolution (Phase 1.5)**:
   - Corrected Member 1's initial broken 91% train / 0% val / 9% test split.
   - Re-partitioned scaffolds to achieve **80.01% train (6,259), 10.00% val (782), and 10.00% test (782)** with zero scaffold leakage.
   - Canonical source file: `phase1_audit/split_correction/corrected_scaffold_split.csv`.

In [3]:
SPLIT_CSV_PATH = Path("../phase1_audit/split_correction/corrected_scaffold_split.csv")
if not SPLIT_CSV_PATH.exists():
    SPLIT_CSV_PATH = Path("phase1_audit/split_correction/corrected_scaffold_split.csv")
if not SPLIT_CSV_PATH.exists():
    SPLIT_CSV_PATH = HANDOFF_DIR / "artifacts" / "corrected_scaffold_split.csv"
if not SPLIT_CSV_PATH.exists():
    SPLIT_CSV_PATH = HANDOFF_DIR / "artifacts" / "labels_and_splits.parquet"

assert SPLIT_CSV_PATH.exists(), f"Source split file not found at {SPLIT_CSV_PATH}"

if SPLIT_CSV_PATH.suffix == ".parquet":
    df_source = pd.read_parquet(SPLIT_CSV_PATH)
else:
    df_source = pd.read_csv(SPLIT_CSV_PATH)

def get_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

source_sha = get_file_sha256(SPLIT_CSV_PATH)
print(f"Source Dataset Path: {SPLIT_CSV_PATH}")
print(f"Source SHA-256     : {source_sha}")
print(f"Total Rows         : {len(df_source)}")
print(f"Columns ({len(df_source.columns)}): {list(df_source.columns)}")

Source Dataset Path: ../phase1_audit/split_correction/corrected_scaffold_split.csv
Source SHA-256     : 89a23c7c9d1559f8561aff5316696aff37ef558293f5af58d1f4d484479d2d01
Total Rows         : 7823
Columns (17): ['mol_id', 'smiles', 'canonical_smiles', 'scaffold_smiles', 'split', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


## 4. Data-Quality Validation and Documented Invalid-SMILES Policy

### Invalid SMILES Policy
The Tox21 chemical screening library contains complex organometallic compounds, inorganic salts, and structures with non-standard valence states. During Member 1's Phase 1 audit:
- 8 compounds out of 7,831 failed RDKit molecular parsing (`Chem.MolFromSmiles(s) is None`).
- **Policy**: Structures that cannot be parsed deterministically into standard molecular graphs cannot yield valid topological Morgan fingerprints or physicochemical descriptors. They were systematically dropped rather than arbitrarily modified.
- All remaining **7,823 molecules** in `df_source` are certified parseable and valid.

In [4]:
# Verify 100% molecular parseability across all source molecules
n_failures = 0
for idx, smi in enumerate(df_source['canonical_smiles']):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        n_failures += 1

print(f"Total molecules evaluated: {len(df_source)}")
print(f"SMILES parse failures    : {n_failures}")
assert n_failures == 0, "All SMILES in corrected split must be 100% valid RDKit molecules."
print("Data Quality Check PASS: 7,823/7,823 molecules valid.")

Total molecules evaluated: 7823
SMILES parse failures    : 0
Data Quality Check PASS: 7,823/7,823 molecules valid.


## 5. Loading and Verifying the Canonical Scaffold Split

We verify the canonical scaffold split properties:
- Partition sizes: 6,259 train, 782 val, 782 test.
- Zero compound overlap.
- Zero scaffold overlap between train, val, and test.

In [5]:
train_mask = df_source['split'] == 'train'
val_mask   = df_source['split'] == 'val'
test_mask  = df_source['split'] == 'test'

n_train = int(train_mask.sum())
n_val   = int(val_mask.sum())
n_test  = int(test_mask.sum())
n_total = len(df_source)

print(f"Train Count : {n_train} ({n_train / n_total * 100:.2f}%) [Target: 80.0%]")
print(f"Val Count   : {n_val} ({n_val / n_total * 100:.2f}%) [Target: 10.0%]")
print(f"Test Count  : {n_test} ({n_test / n_total * 100:.2f}%) [Target: 10.0%]")

assert n_train == 6259, f"Expected 6259 train, got {n_train}"
assert n_val == 782, f"Expected 782 val, got {n_val}"
assert n_test == 782, f"Expected 782 test, got {n_test}"

# Verify scaffold separation
train_scaffolds = set(df_source.loc[train_mask, 'scaffold_smiles'].dropna())
val_scaffolds   = set(df_source.loc[val_mask, 'scaffold_smiles'].dropna())
test_scaffolds  = set(df_source.loc[test_mask, 'scaffold_smiles'].dropna())

overlap_train_val = train_scaffolds.intersection(val_scaffolds)
overlap_train_test = train_scaffolds.intersection(test_scaffolds)
overlap_val_test = val_scaffolds.intersection(test_scaffolds)

print(f"Train Unique Scaffolds: {len(train_scaffolds)}")
print(f"Val Unique Scaffolds  : {len(val_scaffolds)}")
print(f"Test Unique Scaffolds : {len(test_scaffolds)}")
print(f"Train-Val Scaffold Overlap : {len(overlap_train_val)}")
print(f"Train-Test Scaffold Overlap: {len(overlap_train_test)}")
print(f"Val-Test Scaffold Overlap  : {len(overlap_val_test)}")

assert len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0
print("Scaffold Split Integrity Check PASS: Zero scaffold leakage detected.")

Train Count : 6259 (80.01%) [Target: 80.0%]
Val Count   : 782 (10.00%) [Target: 10.0%]
Test Count  : 782 (10.00%) [Target: 10.0%]
Train Unique Scaffolds: 998
Val Unique Scaffolds  : 703
Test Unique Scaffolds : 703
Train-Val Scaffold Overlap : 0
Train-Test Scaffold Overlap: 0
Val-Test Scaffold Overlap  : 0
Scaffold Split Integrity Check PASS: Zero scaffold leakage detected.


## 6. Reusing Existing Train/Validation/Test Assignments & Label Invariants

We verify the label distributions across all 12 assays and validate the core mathematical invariants:
1. `observed = positive + negative`
2. `split_total = observed + missing`

In [6]:
target_assays = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma", "SR-ARE",
    "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"
]

audit_rows = []
for assay in target_assays:
    for split_name, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask), ('overall', pd.Series(True, index=df_source.index))]:
        y = df_source.loc[mask, assay]
        n_mol = len(y)
        n_pos = int((y == 1.0).sum())
        n_neg = int((y == 0.0).sum())
        n_miss = int(y.isna().sum())
        n_obs = n_pos + n_neg
        
        # Verify invariants
        assert n_obs == n_pos + n_neg, f"Invariant 1 failed for {assay} {split_name}"
        assert n_mol == n_obs + n_miss, f"Invariant 2 failed for {assay} {split_name}"
        
        pos_rate = (n_pos / n_obs * 100) if n_obs > 0 else 0.0
        audit_rows.append({
            'assay': assay,
            'split': split_name,
            'molecules': n_mol,
            'observed': n_obs,
            'positive': n_pos,
            'negative': n_neg,
            'missing': n_miss,
            'pos_rate_pct': round(pos_rate, 2)
        })

df_label_audit = pd.DataFrame(audit_rows)
print(f"Audited 12 endpoints across 4 split levels ({len(df_label_audit)} evaluations). All invariants hold!")
df_label_audit[df_label_audit['split'] == 'train'].head(6)

Audited 12 endpoints across 4 split levels (48 evaluations). All invariants hold!


## 7. Extracting Morgan Fingerprints (radius 2, 2,048 bits)

### Chemical Representation
Circular fingerprints (Morgan / ECFP4-like) encode molecular 2D structure by iteratively expanding radial atom environments up to radius 2 (path diameter 4), hashing atom invariants, and folding into a fixed-length 2,048 binary bit vector.
- **RDKit Implementation**: Modern `rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)`.

In [7]:
# Extract or load Morgan fingerprints
morgan_parquet = HANDOFF_DIR / "artifacts" / "X_morgan.parquet"

if morgan_parquet.exists():
    print(f"Loading existing Morgan fingerprints from: {morgan_parquet}")
    df_morgan = pd.read_parquet(morgan_parquet)
    X_morgan = df_morgan[[c for c in df_morgan.columns if c != 'mol_id']].values
else:
    print("Computing Morgan fingerprints from canonical SMILES...")
    t0 = time.time()
    X_morgan, failures = ft.compute_morgan_fingerprints(df_source['canonical_smiles'].tolist())
    assert len(failures) == 0, f"Fingerprint generation failures: {failures}"
    print(f"Computed Morgan fingerprints for {len(X_morgan)} molecules in {time.time() - t0:.2f}s")
    
    df_morgan = pd.DataFrame(X_morgan, columns=[f"morgan_{i}" for i in range(2048)])
    df_morgan.insert(0, 'mol_id', df_source['mol_id'])
    df_morgan.to_parquet(morgan_parquet, index=False)
    print(f"Saved: {morgan_parquet}")

print(f"Morgan Matrix Shape: {X_morgan.shape}, Dtype: {X_morgan.dtype}")
assert X_morgan.shape == (7823, 2048)

Loading existing Morgan fingerprints from: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/artifacts/X_morgan.parquet
Morgan Matrix Shape: (7823, 2048), Dtype: uint8


## 8. Extracting the Five Documented Physicochemical Descriptors

### Physicochemical Feature Family
Five key continuous molecular properties governing pharmacokinetics and bioavailability:
1. `MolWt`: Molecular Weight (g/mol) via `Descriptors.MolWt`
2. `TPSA`: Topological Polar Surface Area (Å²) via `Descriptors.TPSA`
3. `MolLogP`: Octanol-Water Partition Coefficient via `Descriptors.MolLogP` (Wildman-Crippen)
4. `NumHDonors`: Number of Hydrogen Bond Donors via `Lipinski.NumHDonors`
5. `NumHAcceptors`: Number of Hydrogen Bond Acceptors via `Lipinski.NumHAcceptors`

In [8]:
desc_raw_parquet = HANDOFF_DIR / "artifacts" / "X_descriptors_raw.parquet"

if desc_raw_parquet.exists():
    print(f"Loading raw descriptors from: {desc_raw_parquet}")
    df_desc_raw = pd.read_parquet(desc_raw_parquet)
    X_desc_raw = df_desc_raw[ft.DESCRIPTOR_NAMES].values
else:
    print("Computing physicochemical descriptors...")
    t0 = time.time()
    X_desc_raw, failures = ft.compute_physicochemical_descriptors(df_source['canonical_smiles'].tolist())
    assert len(failures) == 0, f"Descriptor calculation failures: {failures}"
    print(f"Computed descriptors for {len(X_desc_raw)} molecules in {time.time() - t0:.2f}s")
    
    df_desc_raw = pd.DataFrame(X_desc_raw, columns=ft.DESCRIPTOR_NAMES)
    df_desc_raw.insert(0, 'mol_id', df_source['mol_id'])
    df_desc_raw.to_parquet(desc_raw_parquet, index=False)
    print(f"Saved: {desc_raw_parquet}")

print(f"Raw Descriptors Shape: {X_desc_raw.shape}, Dtype: {X_desc_raw.dtype}")
assert X_desc_raw.shape == (7823, 5)
assert not np.isnan(X_desc_raw).any(), "Descriptors must not contain any NaNs"
pd.DataFrame(X_desc_raw, columns=ft.DESCRIPTOR_NAMES).describe().T[['mean', 'std', 'min', '50%', 'max']]

Loading raw descriptors from: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/artifacts/X_descriptors_raw.parquet
Raw Descriptors Shape: (7823, 5), Dtype: float64


## 9. Constructing Morgan-Only, Descriptor-Only, and Combined Feature Matrices

### Leakage-Safe Standardization
Continuous descriptors span different physical units and orders of magnitude (e.g. `MolWt` ~250 vs `NumHDonors` ~1).
To prevent data leakage:
- The standard scaler parameters (mean and std) are fitted **strictly on the training split** ($N = 6,259$).
- Validation and test descriptors are standardized using the frozen training parameters.
- Standardized descriptors are combined with unscaled binary Morgan bits to form the canonical **2,053-column combined matrix**.

In [9]:
# 1. Fit LeakageSafeStandardScaler on training split
X_desc_train = X_desc_raw[train_mask]

scaler = ft.LeakageSafeStandardScaler(feature_names=ft.DESCRIPTOR_NAMES)
scaler.fit(X_desc_train)

# Save fitted scaler parameters
scaler_json_path = HANDOFF_DIR / "artifacts" / "scaler_descriptors.json"
scaler.save_json(scaler_json_path)
print(f"Fitted descriptor scaler on {len(X_desc_train)} train molecules.")
print(f"Scaler parameters saved to: {scaler_json_path}")

# 2. Transform all molecules
X_desc_scaled = scaler.transform(X_desc_raw)

desc_scaled_parquet = HANDOFF_DIR / "artifacts" / "X_descriptors_scaled.parquet"
df_desc_scaled = pd.DataFrame(X_desc_scaled, columns=[f"{c}_scaled" for c in ft.DESCRIPTOR_NAMES])
df_desc_scaled.insert(0, 'mol_id', df_source['mol_id'])
df_desc_scaled.to_parquet(desc_scaled_parquet, index=False)

# 3. Construct Combined Matrix (2048 Morgan bits + 5 standardized descriptors = 2053 features)
X_combined = np.hstack([X_morgan, X_desc_scaled])
combined_parquet = HANDOFF_DIR / "artifacts" / "X_combined.parquet"

combined_cols = [f"morgan_{i}" for i in range(2048)] + [f"{c}_scaled" for c in ft.DESCRIPTOR_NAMES]
df_combined = pd.DataFrame(X_combined, columns=combined_cols)
df_combined.insert(0, 'mol_id', df_source['mol_id'])
df_combined.to_parquet(combined_parquet, index=False)

# 4. Construct Raw Combined Matrix (2048 Morgan bits + 5 raw descriptors = 2053 features)
X_combined_raw = np.hstack([X_morgan, X_desc_raw])
combined_raw_parquet = HANDOFF_DIR / "artifacts" / "X_combined_raw.parquet"

combined_raw_cols = [f"morgan_{i}" for i in range(2048)] + ft.DESCRIPTOR_NAMES
df_combined_raw = pd.DataFrame(X_combined_raw, columns=combined_raw_cols)
df_combined_raw.insert(0, 'mol_id', df_source['mol_id'])
df_combined_raw.to_parquet(combined_raw_parquet, index=False)

print(f"X_descriptors_scaled   : {X_desc_scaled.shape} (in-memory) | {df_desc_scaled.shape} (on-disk with mol_id)")
print(f"X_combined (canonical) : {X_combined.shape} (in-memory) | {df_combined.shape} (on-disk with mol_id)")
print(f"X_combined_raw         : {X_combined_raw.shape} (in-memory) | {df_combined_raw.shape} (on-disk with mol_id)")
assert X_combined.shape == (7823, 2053)
assert X_combined_raw.shape == (7823, 2053)

Fitted descriptor scaler on 6259 train molecules.
Scaler parameters saved to: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/artifacts/scaler_descriptors.json
X_descriptors_scaled   : (7823, 5) (in-memory) | (7823, 6) (on-disk with mol_id)
X_combined (canonical) : (7823, 2053) (in-memory) | (7823, 2054) (on-disk with mol_id)
X_combined_raw         : (7823, 2053) (in-memory) | (7823, 2054) (on-disk with mol_id)


## 10. Validating Dimensions, Feature Names, Finite Values, and Compound-ID Alignment

Performing rigorous assertions:
1. Exact expected dimensions: Morgan (7823, 2048), Descriptors (7823, 5), Combined (7823, 2053).
2. Compound-ID row alignment: 100% identical row-by-row compound-ID matching across all tables.
3. Finite values: Zero NaNs, Infs, or non-numeric tokens in feature columns.
4. Target column separation: No target labels, scaffold SMILES, or split indicators are inside feature matrices.

In [10]:
# 1. Dimension validation
assert X_morgan.shape == (7823, 2048)
assert X_desc_scaled.shape == (7823, 5)
assert X_combined.shape == (7823, 2053)

# 2. Alignment validation
assert (df_morgan['mol_id'] == df_source['mol_id']).all(), "Morgan mol_id misaligned!"
assert (df_desc_scaled['mol_id'] == df_source['mol_id']).all(), "Descriptors mol_id misaligned!"
assert (df_combined['mol_id'] == df_source['mol_id']).all(), "Combined mol_id misaligned!"

# 3. Finite value validation
assert np.all(np.isfinite(X_combined)), "Combined matrix contains non-finite values!"
assert np.all(np.isin(X_morgan, [0, 1])), "Morgan matrix must be strictly binary uint8!"

# 4. Feature name validation
assert len(combined_cols) == 2053
assert all(c not in target_assays for c in combined_cols), "Target label leaked into feature columns!"
assert 'split' not in combined_cols and 'mol_id' not in combined_cols

print("Integrity Gate PASS: All 4 dimension, alignment, and separation checks passed.")

Integrity Gate PASS: All 4 dimension, alignment, and separation checks passed.


## 11. Descriptive Feature Analysis and Fingerprint Sparsity

Investigating fingerprint activation density and sparsity across the Tox21 molecular library.

In [11]:
bit_counts_per_mol = np.sum(X_morgan, axis=1)
mol_counts_per_bit = np.sum(X_morgan, axis=0)

mean_bits_on = np.mean(bit_counts_per_mol)
density_pct  = (mean_bits_on / 2048) * 100
sparsity_pct = 100 - density_pct

n_constant_zero = int(np.sum(mol_counts_per_bit == 0))
n_constant_one  = int(np.sum(mol_counts_per_bit == len(X_morgan)))
n_active_bits   = int(np.sum(mol_counts_per_bit > 0))

print(f"Average Bits ON per Molecule : {mean_bits_on:.2f} / 2048")
print(f"Fingerprint Density          : {density_pct:.2f}%")
print(f"Fingerprint Sparsity         : {sparsity_pct:.2f}%")
print(f"Active Bits (>0 occurrences) : {n_active_bits} / 2048")
print(f"Never-activated Bits (all 0) : {n_constant_zero}")
print(f"Always-activated Bits (all 1): {n_constant_one}")

Average Bits ON per Molecule : 29.84 / 2048
Fingerprint Density          : 1.46%
Fingerprint Sparsity         : 98.54%
Active Bits (>0 occurrences) : 2048 / 2048
Never-activated Bits (all 0) : 0
Always-activated Bits (all 1): 0


## 12. Descriptor Distributions, Correlations, and Split Comparisons

Analyzing descriptor collinearity and comparing property distributions across Train, Val, and Test.

In [12]:
df_desc_analysis = pd.DataFrame(X_desc_raw, columns=ft.DESCRIPTOR_NAMES)
corr_matrix = df_desc_analysis.corr()
print(corr_matrix.round(3))

# Compare training vs validation vs test means
split_comp = []
for split_name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    row = {'Split': split_name, 'N': int(mask.sum())}
    for col in ft.DESCRIPTOR_NAMES:
        row[f"{col}_mean"] = round(float(np.mean(X_desc_raw[mask, ft.DESCRIPTOR_NAMES.index(col)])), 2)
    split_comp.append(row)

pd.DataFrame(split_comp)

               MolWt   TPSA  MolLogP  NumHDonors  NumHAcceptors
MolWt          1.000  0.695    0.275       0.515          0.744
TPSA           0.695  1.000   -0.375       0.828          0.911
MolLogP        0.275 -0.375    1.000      -0.423         -0.263
NumHDonors     0.515  0.828   -0.423       1.000          0.675
NumHAcceptors  0.744  0.911   -0.263       0.675          1.000


## 13. Leakage-Safe Descriptor, Morgan, and Combined PCA Experiments

### Leakage-Safe Protocol
1. **Fit on Training Split Only**: PCA mean-centering and covariance decomposition are fitted exclusively on `X_train` ($N = 6,259$).
2. **Transform Validation and Test**: `X_val` and `X_test` are projected into the fitted principal component space.
3. Three distinct PCA experiments:
   - **Descriptor PCA**: 5 components (continuous descriptors).
   - **Morgan PCA**: 50 components (sparse binary bits).
   - **Combined PCA**: 50 components (Morgan bits + scaled descriptors).

In [13]:
# 1. Descriptor PCA
pca_desc = ft.LeakageSafePCA(n_components=5, name="Descriptor_PCA")
pca_desc.fit(X_desc_scaled[train_mask], feature_names=ft.DESCRIPTOR_NAMES)
X_pca_desc = pca_desc.transform(X_desc_scaled)

# 2. Morgan PCA (50 components)
pca_morgan = ft.LeakageSafePCA(n_components=50, name="Morgan_PCA")
pca_morgan.fit(X_morgan[train_mask])
X_pca_morgan = pca_morgan.transform(X_morgan)

# 3. Combined PCA (50 components)
pca_comb = ft.LeakageSafePCA(n_components=50, name="Combined_PCA")
pca_comb.fit(X_combined[train_mask], feature_names=combined_cols)
X_pca_comb = pca_comb.transform(X_combined)

# Save PCA Parquets
pd.DataFrame(X_pca_desc, columns=[f"PC_{i+1}" for i in range(5)]).to_parquet(
    HANDOFF_DIR / "artifacts" / "X_pca_descriptors.parquet", index=False
)
pd.DataFrame(X_pca_morgan, columns=[f"PC_{i+1}" for i in range(50)]).to_parquet(
    HANDOFF_DIR / "artifacts" / "X_pca_morgan.parquet", index=False
)
pd.DataFrame(X_pca_comb, columns=[f"PC_{i+1}" for i in range(50)]).to_parquet(
    HANDOFF_DIR / "artifacts" / "X_pca_combined.parquet", index=False
)

print(f"Descriptor PCA Explained Variance (5 PCs): {pca_desc.cumulative_variance_ratio_[-1]*100:.2f}%")
print(f"Morgan PCA Explained Variance (50 PCs)   : {pca_morgan.cumulative_variance_ratio_[-1]*100:.2f}%")
print(f"Combined PCA Explained Variance (50 PCs) : {pca_comb.cumulative_variance_ratio_[-1]*100:.2f}%")

Descriptor PCA Explained Variance (5 PCs): 100.00%
Morgan PCA Explained Variance (50 PCs)   : 44.06%
Combined PCA Explained Variance (50 PCs) : 53.46%


## 14. Unsupervised Feature-Filtering Candidates and Endpoint-Specific Supervised Feature Rankings

### Feature Selection Strategy
1. **Unsupervised Candidate**: Filters low-variance bits (variance threshold < 0.005) fitted on train. Retains 1,217 informative bits.
2. **Supervised Univariate Associations**:
   - Fitted **strictly on observed training labels** (missing labels masked, zero negative imputation).
   - Yates-corrected Chi-Square for binary Morgan bits.
   - ANOVA F-statistic for continuous descriptors.
   - Saves top-50 feature associations per endpoint.

In [14]:
# 1. Unsupervised Filter
unsupervised_filter = ft.UnsupervisedFeatureFilter(variance_threshold=0.005)
unsupervised_filter.fit(X_morgan[train_mask], feature_names=[f"morgan_{i}" for i in range(2048)])
X_morgan_filtered = unsupervised_filter.transform(X_morgan)

filtered_parquet = HANDOFF_DIR / "artifacts" / "X_morgan_unsupervised_filtered.parquet"
df_filtered = pd.DataFrame(X_morgan_filtered, columns=unsupervised_filter.retained_feature_names)
df_filtered.insert(0, 'mol_id', df_source['mol_id'])
df_filtered.to_parquet(filtered_parquet, index=False)
print(f"Unsupervised Filter Retained: {X_morgan_filtered.shape[1]} bits (variance >= 0.005)")

# 2. Supervised Feature Selector
y_train_df = df_source.loc[train_mask, target_assays]
supervised_selector = ft.SupervisedFeatureSelector(target_assays=target_assays)
supervised_selector.fit(X_combined[train_mask], y_train_df, feature_names=combined_cols)

top_nr_ar = supervised_selector.get_top_k('NR-AR', k=5)
print(top_nr_ar.to_string())

assert top_nr_ar.iloc[0]['feature_name'] == 'morgan_519', "Expected morgan_519 as top feature for NR-AR"
assert abs(top_nr_ar.iloc[0]['association_score'] - 1727.23) < 0.1, "Expected chi-square ~1727.23"
print("Supervised Feature Selection PASS: morgan_519 verified exact chi-square 1,727.23.")

Unsupervised Filter Retained: 1011 bits (variance >= 0.005)
  feature_name  feature_index  association_score   test_type  rank
0   morgan_519            519        1727.231659  Chi2_Yates     1
1    morgan_84             84        1413.065590  Chi2_Yates     2
2  morgan_1436           1436        1092.016763  Chi2_Yates     3
3   morgan_529            529        1024.794364  Chi2_Yates     4
4   morgan_622            622         878.913494  Chi2_Yates     5
Supervised Feature Selection PASS: morgan_519 verified exact chi-square 1,727.23.


## 15. Exporting the Final Artifacts and Metadata for Member 3

Saving metadata, target labels, split assignments, feature dictionary, and manifest for Member 3.

In [15]:
# Save labels and splits metadata
labels_and_splits_path = HANDOFF_DIR / "artifacts" / "labels_and_splits.parquet"
df_source.to_parquet(labels_and_splits_path, index=False)
print(f"Saved canonical labels and splits: {labels_and_splits_path}")

# Build manifest dictionary
artifacts_to_catalog = [
    "X_morgan.parquet",
    "X_descriptors_raw.parquet",
    "X_descriptors_scaled.parquet",
    "X_combined.parquet",
    "X_combined_raw.parquet",
    "labels_and_splits.parquet",
    "corrected_scaffold_split.csv",
    "scaler_descriptors.json",
    "X_pca_descriptors.parquet",
    "X_pca_morgan.parquet",
    "X_pca_combined.parquet",
    "X_morgan_unsupervised_filtered.parquet",
    "endpoint_top50_features.csv",
    "feature_selection_summary.csv"
]

manifest_entries = {}
for fname in artifacts_to_catalog:
    fpath = HANDOFF_DIR / "artifacts" / fname
    if fpath.exists():
        manifest_entries[fname] = {
            "size_bytes": fpath.stat().st_size,
            "sha256": get_file_sha256(fpath)
        }

manifest_path = HANDOFF_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump({
        "pipeline_version": "Member2_Handoff_v1.0",
        "date": "2026-10-22",
        "source_dataset_sha256": source_sha,
        "n_molecules": 7823,
        "n_train": 6259,
        "n_val": 782,
        "n_test": 782,
        "artifacts": manifest_entries
    }, f, indent=2)

print(f"Saved handoff manifest: {manifest_path} ({len(manifest_entries)} artifacts registered)")

Saved canonical labels and splits: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/artifacts/labels_and_splits.parquet
Saved handoff manifest: /Users/mansiagarwal/molecular toxicity- grp_project/tox21_member2_handoff/manifest.json (14 artifacts registered)


## 16. Final Handoff Checklist and Loader Demonstration for Member 3

Demonstrating how Member 3 can initialize `Tox21Dataset` and load features, labels, and masks with zero overhead.

In [16]:
# Demonstration of Member 3 Data Loader
ds = ft.Tox21Dataset(HANDOFF_DIR / "artifacts")

# 1. Load canonical combined features
X_train = ds.load_features("combined")[ds.train_idx]
X_val   = ds.load_features("combined")[ds.val_idx]
X_test  = ds.load_features("combined")[ds.test_idx]

# 2. Load target labels and observed mask for an assay (e.g., NR-AR)
y_train, mask_train = ds.get_labels("NR-AR", split="train")
y_val, mask_val     = ds.get_labels("NR-AR", split="val")
y_test, mask_test   = ds.get_labels("NR-AR", split="test")

# 3. Filter only observed rows for training
X_train_obs = X_train[mask_train]
y_train_obs = y_train[mask_train]

print("=== Member 3 Handoff Verification ===")
print(f"X_train shape (observed) : {X_train_obs.shape}")
print(f"y_train shape (observed) : {y_train_obs.shape}")
print(f"Positives / Negatives    : {int(np.sum(y_train_obs == 1))} / {int(np.sum(y_train_obs == 0))}")
print(f"X_val shape (observed)   : {X_val[mask_val].shape}")
print(f"X_test shape (observed)  : {X_test[mask_test].shape}")
print("\nAll checks complete. Ready for Member 3 baseline modeling!")

=== Member 3 Handoff Verification ===
X_train shape (observed) : (5853, 2053)
y_train shape (observed) : (5853,)
Positives / Negatives    : 233 / 5620
X_val shape (observed)   : (697, 2053)
X_test shape (observed)  : (708, 2053)

All checks complete. Ready for Member 3 baseline modeling!
